# 09 · Самостоятельный эксперимент: точки, дескрипторы, совпадения
**Вопрос:** какой фильтр сохраняет пространственно распределенные правильные совпадения при заданном качестве? Вы реализуете пять компонентов, проверите их на ручных примерах, затем сравните ORB, SIFT и AKAZE на одинаковых парах.

План: 55 минут в аудитории на TODO 1–3 и первые проверки; еще 90–120 минут дома на оценщик, сравнение и отчет. CPU, без обучения/GPU/весов. Полный вычислительный проход обычно занимает минуты; измерьте на своей машине. Пустые TODO намеренно останавливают шаблон. Этот ноутбук содержит весь вспомогательный код.

Перед кодом запишите гипотезу о ratio/mutual и заранее ожидаемое различие ORB/SIFT/AKAZE. Не выбирайте лучшую модель по test.

In [ ]:
import importlib.util, subprocess, sys
requirements={'numpy':'numpy>=1.26,<3','cv2':'opencv-python-headless==4.12.0.88','matplotlib':'matplotlib>=3.8,<4','ipywidgets':'ipywidgets>=8,<9','threadpoolctl':'threadpoolctl>=3.5,<4'}
missing=[package for module,package in requirements.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import cv2
# В текущем Colab штатные варианты OpenCV конфликтуют: cv2 5 не содержит AKAZE.
# Исправляем только одноразовую среду Colab; локальную среду не меняем.
try:
    import google.colab as _colab_runtime
except ImportError:
    _colab_runtime=None
if _colab_runtime is not None and (cv2.__version__.split('.')[0]!='4' or not hasattr(cv2,'AKAZE_create')):
    import importlib.metadata as _opencv_metadata
    _variants=[]
    for _variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try:_opencv_metadata.version(_variant);_variants.append(_variant)
        except _opencv_metadata.PackageNotFoundError:pass
    if _variants:subprocess.check_call([sys.executable,'-m','pip','uninstall','-y',*_variants])
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps','opencv-python-headless==4.12.0.88'])
    raise RuntimeError('OpenCV 4.12 установлен. В Colab выберите «Среда выполнения → Перезапустить сеанс», затем выполните ноутбук сверху.')
if not hasattr(cv2,'AKAZE_create'):
    import importlib.metadata as meta
    installed=[]
    for variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try: meta.version(variant); installed.append(variant)
        except meta.PackageNotFoundError: pass
    if len(installed)==1:
        instruction=f'В отдельной ячейке выполните %pip install {installed[0]}==4.12.0.88 и перезапустите ядро. Сохраняйте только этот вариант OpenCV.'
    else:
        instruction='Найдено несколько вариантов OpenCV либо нестандартная сборка. Создайте чистую среду с одним вариантом opencv-python-headless==4.12.0.88; затем выберите ее kernel.'
    raise RuntimeError('Нужен OpenCV 4.x с AKAZE. '+instruction)
from threadpoolctl import threadpool_limits
_thread_limit=threadpool_limits(limits=1)
try:
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError: pass
%matplotlib inline

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import time, json, csv, hashlib, tempfile, os, urllib.request, platform
cv2.setNumThreads(1)
plt.rcParams.update({'figure.dpi': 105, 'font.size': 10})
assert all(hasattr(cv2, name) for name in ['SIFT_create','ORB_create','AKAZE_create'])
PHOTO_URL='https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/astronaut.png'
PHOTO_SHA='88431cd9653ccd539741b555fb0a46b61558b301d4110412b5bc28b5e3ea6cb5'

def load_photo():
    """NASA photograph, public domain; cached outside the notebook directory."""
    p=Path(os.environ.get('INTEREST_PHOTO_PATH', str(Path(tempfile.gettempdir())/'applied-ai-interest-points'/'astronaut.png')))
    if not p.exists():
        p.parent.mkdir(parents=True,exist_ok=True)
        with urllib.request.urlopen(PHOTO_URL,timeout=60) as response: data=response.read()
        if hashlib.sha256(data).hexdigest()!=PHOTO_SHA: raise ValueError('Download SHA-256 mismatch')
        p.write_bytes(data)
    if hashlib.sha256(p.read_bytes()).hexdigest()!=PHOTO_SHA: raise ValueError('Cached image SHA-256 mismatch')
    img=cv2.imdecode(np.frombuffer(p.read_bytes(),np.uint8),cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img,(384,384),interpolation=cv2.INTER_AREA)

def facade(seed=0,repeated=False,size=384):
    """Authored planar synthetic facade, not a photograph or a model prediction."""
    rng=np.random.default_rng(seed); im=np.full((size,size),190,np.uint8)
    for y in range(24,size-30,48):
        for x in range(24,size-30,48):
            cv2.rectangle(im,(x,y),(x+26,y+30),40,2)
            cv2.line(im,(x+13,y),(x+13,y+30),60,2)
            cv2.line(im,(x,y+15),(x+26,y+15),60,2)
            if not repeated:
                cv2.circle(im,(x+int(rng.integers(4,23)),y+int(rng.integers(4,27))),int(rng.integers(2,6)),int(rng.integers(0,150)),-1)
    if not repeated:
        noise=rng.normal(0,4,im.shape)
        im=np.uint8(np.clip(im+noise,0,255))
        cv2.putText(im,'A'+str(seed),(14,size-10),cv2.FONT_HERSHEY_SIMPLEX,.65,30,2)
    return im

def make_pair(im,angle=12.,scale=.92,dx=6.,dy=-3.,perspective=.00015,blur=.5,gain=1.):
    h,w=im.shape; A=np.vstack([cv2.getRotationMatrix2D(((w-1)/2,(h-1)/2),angle,scale),[0,0,1]])
    A[:2,2]+=[dx,dy]
    P=np.array([[1,0,0],[0,1,0],[perspective,-.6*perspective,1.]])
    H=P@A; H/=H[2,2]
    target=cv2.warpPerspective(im,H,(w,h),flags=cv2.INTER_LINEAR,borderValue=128)
    if blur>0: target=cv2.GaussianBlur(target,(0,0),blur)
    target=np.uint8(np.clip(target.astype(float)*gain,0,255))
    return im,target,H

def extract(im,method,budget=600):
    if method=='ORB': detector=cv2.ORB_create(nfeatures=budget,WTA_K=2);metric='hamming'
    elif method=='SIFT': detector=cv2.SIFT_create(nfeatures=budget);metric='l2'
    elif method=='AKAZE': detector=cv2.AKAZE_create();metric='hamming' # default MLDB
    else: raise ValueError(method)
    kp,desc=detector.detectAndCompute(im,None)
    order=sorted(range(len(kp)),key=lambda i:(-kp[i].response,kp[i].pt[0],kp[i].pt[1]))[:budget]
    pts=np.array([kp[i].pt for i in order],float).reshape(-1,2)
    if desc is None: desc=np.empty((0,detector.descriptorSize()),np.float32 if metric=='l2' else np.uint8)
    else: desc=desc[order]
    return pts,desc,metric

def draw_matches(a,b,p,q,pairs,correct=None,title='',limit=75,path=None):
    h,w=a.shape; canvas=np.hstack([a,b]);fig,ax=plt.subplots(figsize=(11,5),constrained_layout=True)
    ax.imshow(canvas,cmap='gray',vmin=0,vmax=255)
    for k,(i,j) in enumerate(pairs[:limit]):
        color='#10b981' if correct is None or correct[k] else '#f43f5e'
        ax.plot([p[i,0],q[j,0]+w],[p[i,1],q[j,1]],color=color,lw=.65,alpha=.8)
        ax.scatter([p[i,0],q[j,0]+w],[p[i,1],q[j,1]],c=color,s=5)
    ax.set_title(title+' | green: GT error ≤ 3 px; red: incorrect');ax.axis('off')
    if path: fig.savefig(path,dpi=140)
    plt.show();plt.close(fig)

def write_csv(path,rows):
    with Path(path).open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
print('OpenCV',cv2.__version__,'CPU threads',cv2.getNumThreads())

## TODO 1. `project_points`

Однородные координаты: домножьте строку `[x,y,1]` на `H.T`, затем делите на третью компоненту. Нельзя молча заменять нулевой знаменатель на единицу.

**Свой пример:** до реализации придумайте вход из 2–4 точек/дескрипторов, вручную запишите ожидаемый ответ и объясните крайний случай. Открытые тесты ниже не заменяют ваш пример.

In [ ]:
def project_points(points,H):
    """Nx2 xy → Nx2 under x'~H x; return (projected, valid). For |w|<=1e-12 or nonfinite homogeneous result use NaN row/False. Empty -> (0,2)."""
    raise NotImplementedError('TODO 1: homogeneous projection')

## TODO 2. `descriptor_distances`

Самостоятельно реализуйте матрицу расстояний без BFMatcher. Для Hamming удобна таблица popcount из 256 чисел; не разворачивайте весь массив N×M×D×8. Для L2 приведите тип к float до вычитания. Квадрат L2 потребовал бы квадрата ratio.

**Свой пример:** до реализации придумайте вход из 2–4 точек/дескрипторов, вручную запишите ожидаемый ответ и объясните крайний случай. Открытые тесты ниже не заменяют ваш пример.

In [ ]:
def descriptor_distances(a,b,metric):
    """(N,D),(M,D) -> (N,M) distances. l2 = true Euclidean distance (not squared); hamming = number of different bits in packed uint8. Empty sets work; mismatched width/dtype/metric -> ValueError."""
    raise NotImplementedError('TODO 2: distance matrix')

## TODO 3. `ratio_mutual`

Не делите d1 на d2: двойной ноль должен отклоняться строгим неравенством. Mutual проверяется по исходной матрице расстояний, а не по уже отфильтрованному списку.

**Свой пример:** до реализации придумайте вход из 2–4 точек/дескрипторов, вручную запишите ожидаемый ответ и объясните крайний случай. Открытые тесты ниже не заменяют ваш пример.

In [ ]:
def ratio_mutual(D,ratio=.75,mutual=True):
    """Distance matrix -> Kx2 integer [query,train]. Strict d1<ratio*d2; stable lower-index tie choice. If <2 targets, no pairs. Mutual: query must be first argmin of that target column. Reject invalid ratio/distances via ValueError."""
    raise NotImplementedError('TODO 3: ratio and mutual')

## TODO 4. `grid_coverage`

Покрытие — доля занятых ячеек, а не доля площади выпуклой оболочки. Точка x=width вне изображения. Эта метрика помогает обнаружить сгущение, но сама не доказывает устойчивость H.

**Свой пример:** до реализации придумайте вход из 2–4 точек/дескрипторов, вручную запишите ожидаемый ответ и объясните крайний случай. Открытые тесты ниже не заменяют ваш пример.

In [ ]:
def grid_coverage(points,shape,grid=4):
    """Fraction of occupied grid×grid source cells. shape=(height,width), coordinates=(x,y). Count each cell once; discard out-of-bounds/nonfinite points. Bounds are 0<=x<width and 0<=y<height."""
    raise NotImplementedError('TODO 4: spatial coverage')

## TODO 5. `evaluate`

Используйте известную H только в оценщике. Все принятые пары входят в precision, в том числе ошибочно принятые невидимые источники. Число visible_queries зависит от детектора, поэтому matching_score нельзя выдавать за recall на общем фиксированном списке GT точек.

**Свой пример:** до реализации придумайте вход из 2–4 точек/дескрипторов, вручную запишите ожидаемый ответ и объясните крайний случай. Открытые тесты ниже не заменяют ваш пример.

In [ ]:
def evaluate(p,q,pairs,H,shape,threshold=3.):
    """Return (metrics, correct_mask, error_per_pair). A source query is visible if finite H projection lies in target. Correct iff visible and Euclidean reprojection error<=threshold. precision=correct/all pairs; matching_score=unique correct queries/visible queries; empty denominator ->0. repeatability=fraction visible source points with ANY target detector point within threshold (many-to-one diagnostic). coverage=grid_coverage of unique correct source points. Include matches,good,visible_queries,precision,matching_score,repeatability,coverage,median_error of finite errors (None if none finite). Invalid indices -> ValueError."""
    raise NotImplementedError('TODO 5: shared evaluator')

## Открытые тесты

Все они должны пройти. Добавьте свои проверки рядом; тестируйте не только типичный вход, но и пустые данные, равные расстояния и границы кадра.

In [ ]:
# Open tests: expected answers are small and independent of OpenCV detectors.
p,v=project_points([[1,2]],np.array([[2,0,3],[0,3,-1],[0,0,1.]]))
assert np.allclose(p,[[5,5]]) and v.tolist()==[True]
p,v=project_points([[0,0],[2,0]],np.array([[1,0,0],[0,1,0],[1,0,0.]]))
assert not v[0] and np.isnan(p[0]).all() and np.allclose(p[1],[1,0])
assert project_points([],np.eye(3))[0].shape==(0,2)
assert np.array_equal(descriptor_distances(np.uint8([[0],[255]]),np.uint8([[1],[128]]),'hamming'),[[1,1],[7,7]])
assert np.allclose(descriptor_distances([[0,0],[3,4]],[[0,0]],'l2'),[[0],[5]])
assert descriptor_distances(np.empty((0,2)),np.empty((3,2)),'l2').shape==(0,3)
# Hamming: 0 -> 128 is one bit; byte-L2 would incorrectly prefer 3.
assert descriptor_distances(np.uint8([[0]]),np.uint8([[3],[128]]),'hamming').argmin()==1
assert ratio_mutual([[1,1],[0,0]],.9).shape==(0,2)
assert ratio_mutual(np.ones((2,1)),.8).shape==(0,2)
assert np.array_equal(ratio_mutual([[1,4],[2,5]],.8,True),[[0,0]])
assert len(ratio_mutual([[1,4],[2,5]],.8,False))==2
assert len(ratio_mutual([[2,4]],.5,False))==0 # strict inequality
assert np.isclose(grid_coverage([[0,0],[0,0],[99,99],[100,100],[-1,2]],(100,100)),2/16)
p=np.array([[10,10],[20,20],[120,10.]])
r,c,e=evaluate(p,np.array([[13,14],[20,20],[120,10]]),[[0,0],[1,1],[2,2]],np.eye(3),(100,100),3.)
assert r['good']==1 and r['visible_queries']==2 and np.isclose(r['precision'],1/3) and r['matching_score']==.5
assert c.tolist()==[False,True,False] and np.isclose(e[0],5.)
r,_,_=evaluate(np.empty((0,2)),np.empty((0,2)),[],np.eye(3),(10,10))
assert r['precision']==r['matching_score']==r['repeatability']==0 and r['median_error'] is None
for bad in [np.array([[-1,2.]]),np.array([[np.nan,2.]])]:
    try:ratio_mutual(bad);raise AssertionError('Expected ValueError')
    except ValueError:pass
print('Open tests passed: projection, metric, ties/empty input, mutual, boundaries and denominator.')
# A point at infinity is not visible and must not generate a nonstandard NaN JSON number.
r,c,e=evaluate([[0,0]],[[1,1]],[[0,0]],np.array([[1,0,0],[0,1,0],[1,0,0]]),(100,100))
assert r['visible_queries']==0 and r['good']==0 and r['median_error'] is None
assert ratio_mutual(np.empty((0,0))).shape==(0,2)
assert ratio_mutual(np.empty((3,0))).shape==(0,2)
assert len(ratio_mutual([[0,2]],.75))==1

## Собственный контрольный пример и объяснение
Запишите ожидаемые значения до запуска. В своем примере обеспечьте одну верную пару, одну неоднозначную и одну геометрически ложную; покажите, что descriptor distance и geometric error имеют разные единицы и роли. Затем вызовите свои функции. Объясните, какой результат изменится при перестановке строк/столбцов матрицы расстояний.

In [ ]:
# TODO: ваш контрольный пример (не копия открытых тестов).

## Общий протокол и val
Размер 384×384, максимум 600 точек с наибольшим откликом на каждом изображении; реальное число печатается в visible_queries и matches. ORB использует WTA_K=2, AKAZE — стандартный MLDB, SIFT — float descriptor. В основной части matcher — ваша матрица расстояний, поэтому время включает учебный NumPy matcher.

Val: 3 преобразования NASA фото + 3 независимо сгенерированных фасада. Test: 3 других преобразования того же фото + 3 новых seeds фасада. Это разделение **преобразований**, не независимых реальных сцен. Train не нужен: методы не дообучаются. Синтетика дает точную H, контролируемую повторяемость текстуры и известный предел применимости.

Общий оценщик: ошибка переноса ≤3 px, pooled precision, matching score, направленная точечная повторяемость (без сравнения масштабов областей поддержки) и покрытие верными точками исходной сетки 4×4. Преобразования и порог фиксированы для всех. Ratio выбирается из {.60,.75,.90} отдельно для каждого метода при mutual=True. Отдельно посмотрите реальные и синтетические строки, не ограничивайтесь общей средней.

**Правило выбора объявлено заранее:** pooled val precision≥.90; затем максимальное среднее покрытие, затем число правильных пар; равенство разрешается меньшим ratio. Если нет допустимого кандидата, сообщите это и изменяйте протокол только по val. Различие числа детектированных точек остается частью сравнения полных конвейеров; оно не позволяет сравнить одни дескрипторы изолированно.

In [ ]:
RESULTS=Path('results');RESULTS.mkdir(exist_ok=True)
METHODS=['ORB','SIFT','AKAZE'];RATIOS=[.6,.75,.9];BUDGET=600
PHOTO=load_photo()
# Three real-image transformations and three independent synthetic facades per split.
# Real-image val/test share the source photo: held-out transformations, not held-out scenes.
def cases(split):
    if split=='val': specs=[(11,.94,.2,1.),(-18,.85,.8,.85),(28,1.06,1.1,1.1)];seeds=[101,102,103]
    elif split=='test': specs=[(-30,.8,.6,.9),(20,.94,1.5,1.15),(8,1.08,.3,.75)];seeds=[201,202,203]
    else: raise ValueError(split)
    for k,(angle,scale,blur,gain) in enumerate(specs):
        for source in ['real','synthetic']:
            im=PHOTO if source=='real' else facade(seeds[k])
            pair=make_pair(im,angle,scale,dx=4+2*k,dy=-3-k,perspective=.0001*(k+1),blur=blur,gain=gain)
            yield f'{source}_{k}',pair

def detect_distance(a,b,method):
    start=time.perf_counter();p,d1,metric=extract(a,method,BUDGET);q,d2,_=extract(b,method,BUDGET)
    detect_ms=(time.perf_counter()-start)*1000
    start=time.perf_counter();D=descriptor_distances(d1,d2,metric);distance_ms=(time.perf_counter()-start)*1000
    return p,q,D,detect_ms,distance_ms

def summarize(rows):
    n=sum(r['matches'] for r in rows);good=sum(r['good'] for r in rows)
    return {'pooled_precision':good/n if n else 0.,'mean_good':float(np.mean([r['good'] for r in rows])),
      'mean_coverage':float(np.mean([r['coverage'] for r in rows])),
      'mean_matching_score':float(np.mean([r['matching_score'] for r in rows]))}

val_rows=[]
for case_id,(a,b,H) in cases('val'):
    for method in METHODS:
        p,q,D,_,_=detect_distance(a,b,method)
        for ratio in RATIOS:
            pairs=ratio_mutual(D,ratio,True);metrics,_,_=evaluate(p,q,pairs,H,a.shape)
            val_rows.append({'case':case_id,'method':method,'ratio':ratio,**metrics})
write_csv(RESULTS/'validation.csv',val_rows)
summary=[]
for method in METHODS:
    for ratio in RATIOS:
        rows=[r for r in val_rows if r['method']==method and r['ratio']==ratio]
        summary.append({'method':method,'ratio':ratio,**summarize(rows)})
write_csv(RESULTS/'validation_summary.csv',summary)
for row in summary: print(row)
fig,axs=plt.subplots(1,2,figsize=(10,3.6),constrained_layout=True)
for method in METHODS:
    rows=[r for r in summary if r['method']==method]
    axs[0].plot(RATIOS,[r['pooled_precision'] for r in rows],'-o',label=method)
    axs[1].plot(RATIOS,[r['mean_coverage'] for r in rows],'-o',label=method)
axs[0].axhline(.9,color='gray',ls='--');axs[0].set(xlabel='Ratio threshold',ylabel='Pooled precision @3 px',ylim=(0,1.05))
axs[1].set(xlabel='Ratio threshold',ylabel='Mean correct source-cell coverage',ylim=(0,1.05))
for ax in axs: ax.legend()
fig.savefig(RESULTS/'validation.png',dpi=140);plt.show();plt.close(fig)

## Интерпретация val перед фиксацией
Запишите: какие настройки допустимы; какой метод/ratio вы бы выбрали по объявленному правилу; где теряются совпадения; почему большой matching_score при малом числе точек еще не гарантирует хорошее покрытие. Проверьте prediction→measurement→explanation для исходной гипотезы.

In [ ]:
# Predeclared selection: per method precision >= .90, then maximize correct cell coverage,
# then mean correct matches; tie-break favors smaller ratio. If infeasible, say so.
selected={}
for method in METHODS:
    eligible=[r for r in summary if r['method']==method and r['pooled_precision']>=.9]
    if not eligible: raise RuntimeError(f'{method}: no val setting satisfies precision>=.90; report this and revise using val only.')
    selected[method]=max(eligible,key=lambda r:(r['mean_coverage'],r['mean_good'],-r['ratio']))
winner=max(METHODS,key=lambda m:(selected[m]['mean_coverage'],selected[m]['mean_good']))
locked={'ratios':{m:selected[m]['ratio'] for m in METHODS},'winner':winner,'budget':BUDGET,
        'threshold_px':3.,'mutual':True,'photo_sha256':PHOTO_SHA,
        'selection_rule':'precision>=.90, then coverage, then good count, then smaller ratio',
        'val':selected,'test_opened':False}
(RESULTS/'selection_locked.json').write_text(json.dumps(locked,indent=2))
print('Frozen using val:',locked['ratios'],'selected method:',winner)

## Test и одна заранее объявленная абляция
После записи вывода val переключите RUN_TEST=True. Сравните все три метода с их **зафиксированными** ratio. Абляция выбранного по val конвейера отключает только mutual. Ее качество измеряется на тех же признаках; время абляции отдельно не объявляется.

Время: один прогрев и три измерения каждой пары; два detect+describe, полная матрица расстояний и фильтр. Не входят загрузка, формирование пары, GT-оценка и рисунок. Используются один OpenCV thread и один BLAS thread. Это время данной реализации и машины; оно не является опубликованным benchmark скорости SIFT/ORB/AKAZE. Сравните median detect/describe и matching отдельно.

На одинаковых `real_0` и `synthetic_1` выводятся все три метода. Найдите успех и ошибку, не меняйте протокол после просмотра test. Добавьте общий трудный пример с сильным размытием или повторяющимися окнами; обозначьте его диагностическим, вне выбора и основного test.

In [ ]:
RUN_TEST=False

In [ ]:
if RUN_TEST:
    config=json.loads((RESULTS/'selection_locked.json').read_text())
    assert config['photo_sha256']==PHOTO_SHA and config['budget']==BUDGET
    rows=[];examples=[];timing_rows=[]
    for case_id,(a,b,H) in cases('test'):
        for method in METHODS:
            ratio=config['ratios'][method]
            # One warm-up and three complete runs; timing excludes I/O, warping, GT and drawing.
            run_times=[]
            for rep in range(4):
                start=time.perf_counter();p,q,D,detect_ms,distance_ms=detect_distance(a,b,method)
                t=time.perf_counter();pairs=ratio_mutual(D,ratio,True);filter_ms=(time.perf_counter()-t)*1000
                total_ms=(time.perf_counter()-start)*1000
                if rep:run_times.append((detect_ms,distance_ms+filter_ms,total_ms))
            metric,correct,err=evaluate(p,q,pairs,H,a.shape)
            med=np.median(run_times,axis=0)
            rows.append({'case':case_id,'method':method,'ratio':ratio,**metric,
                'detect_describe_ms':float(med[0]),'match_ms':float(med[1]),'total_ms':float(med[2])})
            for rep,values in enumerate(run_times):timing_rows.append({'case':case_id,'method':method,'repeat':rep+1,'detect_describe_ms':values[0],'match_ms':values[1],'total_ms':values[2]})
            if case_id in ['real_0','synthetic_1']:
                draw_matches(a,b,p,q,pairs,correct,f'{case_id} · {method} · ratio {ratio} · {metric["good"]}/{metric["matches"]} correct',path=RESULTS/f'{case_id}_{method}.png')
            if method==winner:
                # Predeclared ablation: only remove mutual check; ratio and features unchanged.
                ab=ratio_mutual(D,ratio,False);am,_,_=evaluate(p,q,ab,H,a.shape)
                rows.append({'case':case_id,'method':method+'_no_mutual','ratio':ratio,**am,
                    'detect_describe_ms':None,'match_ms':None,'total_ms':None})
    write_csv(RESULTS/'test.csv',rows);write_csv(RESULTS/'timing_repeats.csv',timing_rows)
    totals=[]
    for method in METHODS+[winner+'_no_mutual']:
        items=[r for r in rows if r['method']==method]
        result={'method':method,**summarize(items)}
        if method in METHODS:
            result.update({k:float(np.median([r[k] for r in items])) for k in ['detect_describe_ms','match_ms','total_ms']})
        totals.append(result)
    (RESULTS/'test_summary.json').write_text(json.dumps(totals,indent=2))
    environment={'python':platform.python_version(),'numpy':np.__version__,'opencv':cv2.__version__,
        'cpu':next((line.split(':',1)[1].strip() for line in Path('/proc/cpuinfo').read_text().splitlines() if line.startswith('model name')),platform.machine()) if Path('/proc/cpuinfo').exists() else (platform.processor() or platform.machine()),'opencv_threads':cv2.getNumThreads(),'feature_budget':BUDGET,
        'image_hw':[384,384],'timing_repeats':3,'warmup':1,'scope':'2x detect/describe + educational NumPy distances + ratio/mutual; no GT/I/O/drawing'}
    (RESULTS/'environment.json').write_text(json.dumps(environment,indent=2))
    config['test_opened']=True;(RESULTS/'selection_locked.json').write_text(json.dumps(config,indent=2))
    for row in totals:print(row)
else:
    print('Test remains closed. Finish val interpretation, freeze parameters, then set RUN_TEST=True.')

## Итоговый отчет
1. Таблица 36 пунктов демо, исходная гипотеза и ваши 5 ручных контрольных примеров.
2. Какие TODO вы проверили, найденная ошибка реализации и ее исправление.
3. Таблица val/test по трем методам: precision, good, coverage, matching_score, repeatability; различие реальных/синтетических строк.
4. Абляция mutual: что изменилось и почему. Не объявляйте причинный эффект за пределами этого протокола.
5. Два одинаковых случая для всех методов, успех/ошибка и дополнительный failure case; подписи с единицами/GT.
6. Медианное время и среда; ограничения одной фотографии и плоской H; какие соответствия вы передадите геометрии камеры в следующей работе.

Сохраните CSV/JSON/рисунки из `results/` для отчета. В Git не кладите скачанное фото/кэш; перед распространением notebook очистите outputs. Объяснение важнее максимального числа совпадений.

## Источники и границы вывода

- [OpenCV: feature matching](https://docs.opencv.org/4.13.0/dc/dc3/tutorial_py_matcher.html): L2 для SIFT, Hamming для ORB (`WTA_K=2`) и используемого AKAZE-MLDB. Детектор выбирает координаты, дескриптор описывает окрестность; это разные этапы.
- [Lowe, 2004](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf): SIFT и отношение расстояний до двух ближайших соседей. Число .75 здесь — вариант для опыта, не универсальный порог.
- [OpenCV: AKAZE](https://docs.opencv.org/4.13.0/db/d70/tutorial_akaze_matching.html): используем стандартный бинарный MLDB.
- [NASA / scikit-image astronaut](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.astronaut): открытое фото Eileen Collins, public domain; закрепленный файл scikit-image v0.25.2 проверяется SHA-256. Фото скачивается во временный кэш, в notebook не встроено. Объект интереса — локальные текстуры, личность не распознается.

Синтетический фасад нарисован учебным кодом; второй кадр фотографии вычислен известной гомографией. Это контролируемый опыт на плоском изображении, а не две реальные фотографии 3D-сцены и не результат HPatches. Он не проверяет параллакс, изменение видимости или независимость новых сцен.